In [ ]:
# pip install fastapi uvicorn requests pyjwt cryptography

import requests

SUPABASE_URL = "https://nebbedzehvrobugqisju.supabase.co" # project_url
SUPABASE_KEY = "sb_publishable_vaLyuBx0wsgZy3snbrx-pQ_2AbTNn_7" # publishable_key

HEADERS = {
    "apikey": SUPABASE_KEY,
    "Content-Type": "application/json",
}

EMAIL = "akashroy2141@gmail.com"
PASSWORD = "mypassword123"


def sign_up():
    url = f"{SUPABASE_URL}/auth/v1/signup"
    payload = {"email": EMAIL, "password": PASSWORD}
    response = requests.post(url, headers=HEADERS, json=payload)
    print("Signup status:", response.status_code)
    print("Signup response:", response.json())
    return response.json()


def sign_in():
    url = f"{SUPABASE_URL}/auth/v1/token?grant_type=password"
    payload = {"email": EMAIL, "password": PASSWORD}
    response = requests.post(url, headers=HEADERS, json=payload)
    print("Login status:", response.status_code)
    data = response.json()
    print("Login response:", data)
    return data


if __name__ == "__main__":
    # print("=== Signing up ===")
    # sign_up()

    print("\n=== Logging in ===")
    login_data = sign_in()

    access_token = login_data.get("access_token")
    if access_token:
        print("\n=== Access token ===")
        print(access_token)
    else:
        print("\nNo access token returned — check the error above.")

=== Signing up ===

=== Logging in ===
Login status: 200
Login response: {'access_token': 'eyJhbGciOiJFUzI1NiIsImtpZCI6ImI0MDlhN2QxLTRkNjEtNGZhZS1iYzNlLTgzYjliNThmOTU5MCIsInR5cCI6IkpXVCJ9.eyJpc3MiOiJodHRwczovL25lYmJlZHplaHZyb2J1Z3Fpc2p1LnN1cGFiYXNlLmNvL2F1dGgvdjEiLCJzdWIiOiJmODllYTRiMy1iZTlmLTQ0ZDEtYjJiNC00ZDM1ZGE0NWVlMDYiLCJhdWQiOiJhdXRoZW50aWNhdGVkIiwiZXhwIjoxNzg0MDg3MjU5LCJpYXQiOjE3ODQwODM2NTksImVtYWlsIjoiYWthc2hyb3kyMTQxQGdtYWlsLmNvbSIsInBob25lIjoiIiwiYXBwX21ldGFkYXRhIjp7InByb3ZpZGVyIjoiZW1haWwiLCJwcm92aWRlcnMiOlsiZW1haWwiXX0sInVzZXJfbWV0YWRhdGEiOnsiZW1haWwiOiJha2FzaHJveTIxNDFAZ21haWwuY29tIiwiZW1haWxfdmVyaWZpZWQiOnRydWUsInBob25lX3ZlcmlmaWVkIjpmYWxzZSwic3ViIjoiZjg5ZWE0YjMtYmU5Zi00NGQxLWIyYjQtNGQzNWRhNDVlZTA2In0sInJvbGUiOiJhdXRoZW50aWNhdGVkIiwiYWFsIjoiYWFsMSIsImFtciI6W3sibWV0aG9kIjoicGFzc3dvcmQiLCJ0aW1lc3RhbXAiOjE3ODQwODM2NTl9XSwic2Vzc2lvbl9pZCI6ImFlNDBiNjIyLTNmMzEtNDk0Yy05MmU5LWFlNTJkNTQwOWE0NiIsImlzX2Fub255bW91cyI6ZmFsc2V9.tgkTLb2U1Zu19_Cok7srnmth_39V9TBvgFnYpI6sO7ze5ETnimD0RWu70sg

In [ ]:
import requests
from fastapi import FastAPI, Depends, HTTPException
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
from pydantic import BaseModel
import jwt
from jwt import PyJWKClient

app = FastAPI(title="Supabase JWT Auth Demo")

# Supabase project config 
SUPABASE_URL = "https://nebbedzehvrobugqisju.supabase.co"
SUPABASE_ANON_KEY = "sb_publishable_vaLyuBx0wsgZy3snbrx-pQ_2AbTNn_7"
JWKS_URL = f"{SUPABASE_URL}/auth/v1/.well-known/jwks.json"
SUPABASE_AUDIENCE = "authenticated"

# Fetches + caches Supabase's public signing keys automatically
jwks_client = PyJWKClient(JWKS_URL)
bearer_scheme = HTTPBearer()


# Request/response models 
class AuthRequest(BaseModel):
    email: str
    password: str


#  Step 1: Signup — proxies to Supabase Auth 
@app.post("/signup")
def signup(payload: AuthRequest):
    url = f"{SUPABASE_URL}/auth/v1/signup"
    headers = {
        "apikey": SUPABASE_ANON_KEY,
        "Content-Type": "application/json",
    }
    response = requests.post(
        url, headers=headers, json={"email": payload.email, "password": payload.password}
    )
    if response.status_code != 200:
        raise HTTPException(status_code=response.status_code, detail=response.json())
    return response.json()


# Step 2: Login — proxies to Supabase Auth, returns the JWT 
@app.post("/login")
def login(payload: AuthRequest):
    url = f"{SUPABASE_URL}/auth/v1/token?grant_type=password"
    headers = {
        "apikey": SUPABASE_ANON_KEY,
        "Content-Type": "application/json",
    }
    response = requests.post(
        url, headers=headers, json={"email": payload.email, "password": payload.password}
    )
    if response.status_code != 200:
        raise HTTPException(status_code=response.status_code, detail=response.json())
    data = response.json()
    return {
        "access_token": data["access_token"],
        "refresh_token": data["refresh_token"],
        "token_type": data["token_type"],
        "expires_in": data["expires_in"],
    }


#  Step 3: Dependency that verifies the Supabase-issued JWT 
def get_current_user(credentials: HTTPAuthorizationCredentials = Depends(bearer_scheme)):
    token = credentials.credentials
    try:
        # Looks at the token's "kid" header, fetches the matching public key from Supabase's JWKS
        signing_key = jwks_client.get_signing_key_from_jwt(token)
        payload = jwt.decode(
            token,
            signing_key.key,
            algorithms=["ES256", "RS256"],  # pin accepted algorithms
            audience=SUPABASE_AUDIENCE,
        )
    except jwt.PyJWTError as e:
        raise HTTPException(status_code=401, detail=f"Invalid or expired token: {e}")
    return payload


# Step 4: Protected route 
@app.get("/me")
def read_profile(user: dict = Depends(get_current_user)):
    return {
        "user_id": user["sub"],
        "email": user.get("email"),
        "role": user.get("role"),
        "issued_at": user.get("iat"),
        "expires_at": user.get("exp"),
    }


#  Public route, for comparison 
@app.get("/")
def root():
    return {"message": "This route is public — no token required."}